## build_dim_metro_environment
Rebuilds `gold.dim_metro_environment` (1:1 static environment dimension) by **full-outer-joining** `silver.fact_fema_hazard_cbsa` (15 hazard cols) + `silver.fact_noaa_climate_cbsa` (13 climate cols) on `geo_key`, renamed to the descriptive Gold names per `silver_gold_column_name_mapping.md` §5/§6. Outer join = a metro in only one source still gets a row (NULLs on the absent side) — honest coverage (design §3.3). `expected_annual_loss_usd` + `population` are retained as the additive base for the deferred region rollup. Full rebuild via `INSERT OVERWRITE`. StepLog + transform_detail_log.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 3                       # position owned by the orchestrator (G4)
SOURCE_FEMA  = f"{SILVER}.fact_fema_hazard_cbsa"
SOURCE_NOAA  = f"{SILVER}.fact_noaa_climate_cbsa"
TARGET_TABLE = f"{GOLD}.dim_metro_environment"
DIM_GEO      = f"{SILVER}.dim_geo"

# (silver source col -> Gold target col), per silver_gold_column_name_mapping §5/§6.
# Order matches the G0 column order in gold_ddl.py (15 FEMA, then 13 NOAA).
FEMA_RENAMES = [
    ("population", "population"),
    ("eal_valt", "expected_annual_loss_usd"),
    ("risk_score", "overall_risk_score"),
    ("sovi_score", "social_vulnerability_score"),
    ("resl_score", "community_resilience_score"),
    ("hrcn_risks", "hurricane_risk_score"),
    ("cfld_risks", "coastal_flood_risk_score"),
    ("ifld_risks", "inland_flood_risk_score"),
    ("trnd_risks", "tornado_risk_score"),
    ("wfir_risks", "wildfire_risk_score"),
    ("erqk_risks", "earthquake_risk_score"),
    ("hail_risks", "hail_risk_score"),
    ("swnd_risks", "strong_wind_risk_score"),
    ("hwav_risks", "heat_wave_risk_score"),
    ("wntw_risks", "winter_weather_risk_score"),
]
NOAA_RENAMES = [
    ("ann_tavg_normal", "avg_annual_temp_f"),
    ("djf_tavg_normal", "avg_winter_temp_f"),
    ("mam_tavg_normal", "avg_spring_temp_f"),
    ("jja_tavg_normal", "avg_summer_temp_f"),
    ("son_tavg_normal", "avg_autumn_temp_f"),
    ("ann_tmax_normal", "avg_annual_high_temp_f"),
    ("ann_tmin_normal", "avg_annual_low_temp_f"),
    ("jja_tmax_normal", "avg_summer_high_temp_f"),
    ("djf_tmin_normal", "avg_winter_low_temp_f"),
    ("ann_prcp_normal", "annual_precipitation_inches"),
    ("ann_snow_normal", "annual_snowfall_inches"),
    ("ann_htdd_normal", "annual_heating_degree_days"),
    ("ann_cldd_normal", "annual_cooling_degree_days"),
]

# Integer ($/count) carries are NOT rounded; every other measure is a DOUBLE
# rounded to 2 dp at write (FEMA publishes scores to the hundredth; the long float
# tails are population-weighted-mean artifacts, not signal). Gold rounds when serving.
NO_ROUND = {"population", "expected_annual_loss_usd"}

In [ ]:
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_dim_metro_environment: step_log_id={step.step_log_id}")

In [ ]:
# Both Silver sources are static, CBSA-grain, keyed by geo_key (already conformed via
# silver.dim_geo). FULL OUTER join so a metro present in only one source still gets a profile
# row (NULLs on the absent side) — honest coverage (design §3.3). geo_key via coalesce since
# either side may be null on an outer row.
try:
    fema = spark.table(SOURCE_FEMA).alias("h")
    noaa = spark.table(SOURCE_NOAA).alias("c")
    fema_n, noaa_n = fema.count(), noaa.count()

    joined = fema.join(noaa, F.col("h.geo_key") == F.col("c.geo_key"), "full_outer")
    staged = joined.select(
        F.coalesce(F.col("h.geo_key"), F.col("c.geo_key")).alias("geo_key"),
        # Round DOUBLE measures to 2 dp; leave the integer ($/count) carries intact.
        *[(F.col(f"h.{src}") if dst in NO_ROUND else F.round(F.col(f"h.{src}"), 2)).alias(dst)
          for src, dst in FEMA_RENAMES],
        *[F.round(F.col(f"c.{src}"), 2).alias(dst) for src, dst in NOAA_RENAMES],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    staged.createOrReplaceTempView("gold_dim_metro_environment_staging")
    step.rows_read = fema_n + noaa_n
    print(f"build_dim_metro_environment: read fema={fema_n:,} noaa={noaa_n:,}")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Full rebuild via INSERT OVERWRITE (design §2.2). Validate: (1) every geo_key resolves in
# dim_geo (the informational FK is NOT enforced, so check it here); (2) post-count equals the
# staged distinct-geo_key count (one row per metro present in either source).
transform_started = datetime.now(timezone.utc)
try:
    staged = spark.table("gold_dim_metro_environment_staging")
    expected = staged.count()
    orphans = staged.join(spark.table(DIM_GEO).select("geo_key"), "geo_key", "left_anti").count()
    if orphans > 0:
        raise AssertionError(
            f"[{TARGET_TABLE}] {orphans:,} geo_key(s) absent from {DIM_GEO} "
            f"(would violate the dim_geo FK)."
        )

    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_dim_metro_environment_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != expected:
        raise AssertionError(
            f"[{TARGET_TABLE}] Row-count mismatch: expected {expected:,} (distinct geo_key "
            f"across the two sources), table now has {post_count:,}."
        )
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id,
        f"{SOURCE_FEMA},{SOURCE_NOAA}", TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_dim_metro_environment: wrote {post_count:,} rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id,
        f"{SOURCE_FEMA},{SOURCE_NOAA}", TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise